# RAPID example_id matching to gauging station names

In [24]:
import geopandas as gpd
import pandas as pd

import plotly.graph_objects as go

import plotly.express as px

In [18]:
p = '/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/'
df_ex = gpd.read_file(p +'hierarchy_examples_filtered_subdaily_manual_updates_final.gpkg')

dfsub = pd.read_parquet(p + f'subdaily_values/BR/subdaily_timeseries.parquet')

In [22]:
stations = df_ex.loc[df_ex['example_ids'] == '3', 'station_key'].to_list()
display(stations)

['BR:3652880', 'BR:3652890']

In [23]:
stat1 = dfsub[dfsub['station_key'] == stations[0]]
stat2 = dfsub[dfsub['station_key'] == stations[1]]

# RAPID Hydrograph Peak selection

In [27]:
p1 = stat1.copy()
p2 = stat2.copy()

tstart = '2023-02-12 00:00:00+0000'
tend   = '2023-02-23 00:00:00+0000'
p1 = p1[(p1['time'] > tstart) & (p1['time'] < tend)]
p2 = p2[(p2['time'] > tstart) & (p2['time'] < tend)]

In [28]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=p1["time"],
    y=p1["discharge"],
    mode="markers",
    name=p1['station_key'].iloc[0],
    hovertemplate=f"{p1['station_key'].iloc[0]}" +"<br>time: %{x}<br>discharge: %{y}<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=p2["time"],
    y=p2["discharge"],
    mode="markers",
    name=f"{p1['station_key'].iloc[0]}",
    hovertemplate=f"{p2['station_key'].iloc[0]}"+"<br>time: %{x}<br>discharge: %{y}<extra></extra>"
))

fig.update_layout(
    title="Discharge over Time",
    xaxis_title="time",
    yaxis_title="discharge"
)

fig.show('browser')

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


Example 03:
- Peak  tstart = '2023-02-12 00:00:00+0000' tend   = '2023-02-23 00:00:00+0000'
- Exact start 2023-02-14T18:00:00Z --> 8 hour window
- Exact end 2023-02-18T16:00:00Z --> 10 hour window

## Normalize peak files to 30 min or one hour for RAPID
- make sure all timesteps are equal

In [26]:
tstart = "2023-02-12 00:00:00+0000"
tend   = "2023-02-23 00:00:00+0000"

p1 = stat1.copy()
p1["time"] = pd.to_datetime(p1["time"], utc=True)

p1 = p1[(p1["time"] > tstart) & (p1["time"] < tend)].copy()
p1 = p1.sort_values("time")

# set time index
p1i = p1.set_index("time")

# 30-minute discharge series
p1_30min = (
    p1i["discharge"]
    .resample("30min")
    .mean()
    .interpolate("time")
    .reset_index()
)

# hourly discharge series
p1_1h = (
    p1i["discharge"]
    .resample("1h")
    .mean()
    .interpolate("time")
    .reset_index()
)

p1_30min.to_csv(p + "subdaily_values/BR/Ex_03_2023_02_30min.csv", index=False)
p1_1h.to_csv(p + "subdaily_values/BR/Ex_03_2023_02_1h.csv", index=False)


# RAPID Peak plot

In [32]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import netcdf_file

exp_dir = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/sarl03_indep")
run_registry = pd.read_csv(exp_dir / "rapid_run_registry.csv")

series_by_state = {}

for row in run_registry.itertuples(index=False):
    if row.status != "ran":
        continue

    qout_path = Path(row.qout_nc)
    prep_manifest_path = Path(row.rapid_prep_dir) / "rapid_prep_manifest.json"

    with netcdf_file(qout_path, "r", mmap=False) as ds:
        rivid = np.array(ds.variables["rivid"].data).copy()
        time = np.array(ds.variables["time"].data).copy()
        qout = np.array(ds.variables["Qout"].data).copy()

    riv = pd.read_csv(Path(row.rapid_prep_dir) / "riv.csv", header=None)[0].to_numpy()
    final_reach_id = int(riv[-1])
    final_idx = int(np.where(rivid == final_reach_id)[0][0])

    ts = pd.DataFrame(
        {
            "time_seconds": time,
            "q_cms": qout[:, final_idx],
        }
    )
    ts["state_id"] = row.state_id
    ts["final_reach_id"] = final_reach_id

    series_by_state[row.state_id] = ts

# example: one state
print(series_by_state["S001_unit_5"].head())

# combine all states into one long table
all_outlet_ts = pd.concat(series_by_state.values(), ignore_index=True)
print(all_outlet_ts.head())


   time_seconds          q_cms     state_id  final_reach_id
0             0  1.582830e-265  S001_unit_5             796
1          1800  1.135766e-256  S001_unit_5             796
2          3600  8.450349e-249  S001_unit_5             796
3          5400  1.485967e-241  S001_unit_5             796
4          7200  8.996051e-235  S001_unit_5             796
   time_seconds          q_cms   state_id  final_reach_id
0             0  1.392432e-265  S000_base             807
1          1800  9.992275e-257  S000_base             807
2          3600  7.435102e-249  S000_base             807
3          5400  1.307553e-241  S000_base             807
4          7200  7.916642e-235  S000_base             807


In [33]:
fig = go.Figure()


for k in series_by_state.keys():
    u = series_by_state[k]
    start = 0
    end = 450
    t = np.asarray(u["time_seconds"][start:end], dtype=np.float64).copy()/(3600)
    q = np.asarray(u["q_cms"][start:end], dtype=np.float64).copy()
    state_name = u['state_id'].iloc[0]
    # plt.scatter(t, q)
    fig.add_trace(go.Scatter(
        x=t,
        y=q,
        mode="markers",
        name=state_name,
        hovertemplate=f"{state_name}" +"<br>time: %{x}<br>discharge: %{y}<extra></extra>"
    ))

# fig.add_trace(go.Scatter(
#     x=p2["time"],
#     y=p2["discharge"],
#     mode="markers",
#     name=f"{p1['station_key'].iloc[0]}",
#     hovertemplate=f"{p2['station_key'].iloc[0]}"+"<br>time: %{x}<br>discharge: %{y}<extra></extra>"
# ))

fig.update_layout(
    title="Discharge over Time",
    xaxis_title="time",
    yaxis_title="discharge"
)

fig.show('browser')

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


# RAPID Hydrograph Analysis

Use this notebook after `RAPID/run_rapid_experiment.py` has finished. It reads the per-state outlet hydrographs and hydrograph metrics written under each state's `rapid/run/` folder.

In [34]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [35]:
experiment_dir = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/sarl03_indep")
run_registry = pd.read_csv(experiment_dir / "rapid_run_registry.csv")
run_registry.head()

,state_id,rapid_prep_dir,rapid_run_dir,qout_nc,status,hydrograph_status,outlet_hydrograph_csv,hydrograph_metrics_csv,hydrograph_metrics_json,event_start_source,...,outlet_excess_volume_m3,outlet_reach_count,outlet_reach_ids,hydrograph_duration_seconds,metric_config,fall_time_seconds,fall_time_50_seconds,fall_time_10_seconds,e_folding_time_seconds,rise_rate_cms_per_hour
0,S000_base,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,7.411542e+08,1,[807],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",78347.600231,41031.381893,72188.065244,52402.698891,14.098274
1,S001_unit_5,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,7.339993e+08,1,[796],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",78030.098474,40761.589460,71882.981155,52098.195036,14.018287
2,S002_unit_7,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,7.174754e+08,1,[798],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",77295.282181,40143.210519,71172.878856,51394.921534,13.832999
3,S003_unit_20,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,7.275680e+08,1,[797],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",77742.348209,40527.595812,71603.697069,51825.655212,13.945848
4,S004_unit_8,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,7.262855e+08,1,[795],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",77683.974696,40482.427499,71547.024876,51771.472403,13.931285


In [37]:
metric_cols = [
    "state_id",
    "event_start_time_utc",
    "peak_time_utc",
    "event_end_time_utc",
    "peak_discharge_cms",
    "peak_excess_cms",
    "peak_excess_to_end_baseline_cms",
    "time_to_peak_seconds",
    "event_duration_seconds",
    "fall_time_seconds",
    "fall_time_50_seconds",
    "fall_time_10_seconds",
    "e_folding_time_seconds",
    "lag_to_inflow_peak_seconds",
    "peak_attenuation_ratio",
]
run_registry[metric_cols].sort_values("peak_discharge_cms", ascending=False).head(28)

,state_id,event_start_time_utc,peak_time_utc,event_end_time_utc,peak_discharge_cms,peak_excess_cms,peak_excess_to_end_baseline_cms,time_to_peak_seconds,event_duration_seconds,fall_time_seconds,fall_time_50_seconds,fall_time_10_seconds,e_folding_time_seconds,lag_to_inflow_peak_seconds,peak_attenuation_ratio
13,S013_unit_10,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.631597,1772.513207,492.061453,460800.0,367200.0,77354.623099,40187.832358,71230.333136,51449.125772,241200.0,0.987177
9,S009_unit_23,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.590991,1805.215840,496.079586,460800.0,367200.0,78389.854445,41063.980415,72228.601618,52445.588395,241200.0,0.987162
2,S002_unit_7,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.590502,1770.623912,491.777661,460800.0,367200.0,77295.282181,40143.210519,71172.878856,51394.921534,241200.0,0.987162
22,S022_unit_16,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.589140,1801.881532,495.580494,460800.0,367200.0,78263.851505,40954.530114,72108.396994,52320.136034,241200.0,0.987161
24,S024_unit_3,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.587166,1820.860597,497.887063,460800.0,367200.0,78851.342678,41440.567009,72671.758093,52878.397216,241200.0,0.987160
1,S001_unit_5,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.578382,1794.340706,494.648880,460800.0,367200.0,78030.098474,40761.589460,71882.981155,52098.195036,241200.0,0.987157
5,S005_unit_12,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.559411,1782.467218,493.179367,460800.0,367200.0,77661.945701,40456.290571,71526.183121,51747.058163,241200.0,0.987150
23,S023_unit_27,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.550368,1772.699009,491.979132,460800.0,367200.0,77359.122799,40202.667571,71234.025259,51458.397529,241200.0,0.987146
6,S006_unit_15,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.546276,1767.564187,491.346384,460800.0,367200.0,77199.340971,40067.885491,71080.237742,51305.930460,241200.0,0.987145
26,S026_unit_25,2023-02-14T19:00:00+00:00,2023-02-20T03:00:00+00:00,2023-02-19T01:00:00+00:00,2591.542060,1812.612424,496.828875,460800.0,367200.0,78596.456058,41235.674584,72426.573096,52638.361765,241200.0,0.987143


In [38]:
def load_state_hydrograph(state_id: str) -> pd.DataFrame:
    row = run_registry.loc[run_registry["state_id"].eq(state_id)].iloc[0]
    hydro = pd.read_csv(row["outlet_hydrograph_csv"])
    hydro["time_utc"] = pd.to_datetime(hydro["time_utc"], utc=True)
    return hydro

state_id = run_registry.loc[run_registry["status"].eq("ran"), "state_id"].iloc[0]
hydro = load_state_hydrograph(state_id)
hydro.head()

,time_seconds,time_utc,q_outlet_cms,q_inflow_cms
0,0.0,2023-02-12 00:00:00+00:00,1.392432e-265,1631.680
1,1800.0,2023-02-12 00:30:00+00:00,9.992275e-257,1628.365
2,3600.0,2023-02-12 01:00:00+00:00,7.435102e-249,1628.365
3,5400.0,2023-02-12 01:30:00+00:00,1.307553e-241,1625.050
4,7200.0,2023-02-12 02:00:00+00:00,7.916642e-235,1625.050


In [39]:
row = run_registry.loc[run_registry["state_id"].eq(state_id)].iloc[0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_inflow_cms"], mode="lines", name="Inflow"))
fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_outlet_cms"], mode="lines", name="Outlet"))
fig.add_vline(x=pd.to_datetime(row["event_start_time_utc"], utc=True), line_dash="dash", line_color="gray")
fig.add_vline(x=pd.to_datetime(row["peak_time_utc"], utc=True), line_dash="dot", line_color="red")
fig.update_layout(title=f"{state_id} outlet hydrograph", xaxis_title="time", yaxis_title="discharge (cms)")
fig.show()

In [40]:
summary = run_registry.copy()
for column in [
    "time_to_peak_seconds",
    "fall_time_seconds",
    "fall_time_50_seconds",
    "fall_time_10_seconds",
    "e_folding_time_seconds",
    "lag_to_inflow_peak_seconds",
]:
    summary[column.replace("_seconds", "_hours")] = pd.to_numeric(summary[column], errors="coerce") / 3600.0

px.scatter(
    summary,
    x="time_to_peak_hours",
    y="peak_discharge_cms",
    hover_name="state_id",
    color="peak_attenuation_ratio",
    title="Peak magnitude vs. time to peak",
)